# Photometry inspection

Read manufacturer photometry, look at it, and check it before it reaches the model.

Angles are degrees from nadir throughout. Zero points straight down, 90 is the horizon, 180 is the
zenith. Light above 90 degrees is uplight, which is the quantity the model exists to estimate.

## What it needs

Only `numpy`, `scipy`, `matplotlib` and `pandas`. It does **not** need the `illum` package, GDAL, or
anything else heavy, so it opens on a plain Python install.

It does need the reader, `AngularPowerDistribution.py`. Keep a copy of that file in the same folder as
this notebook and the notebook will use it. If you are working inside the `noc-illumina` checkout
instead, the notebook falls back to the installed package automatically. Either way the first cell
prints which file it loaded and that file's fingerprint.

## One reader, not two

This notebook parses nothing itself. All reading happens in `AngularPowerDistribution.py`, the same
file the model pipeline uses. To change how a format is read, change that file in the repository and
copy it here again. Never re-implement a reader in a notebook cell.

That rule exists for a reason. `ies_to_lop.ipynb` and `ldt_reader.ipynb`, which this replaces, each
carried their own parsing. Both were wrong for years and nobody could see it. They took a single
C-plane instead of the azimuth average, wrote two decimal places, which erases any uplight below half
a percent of peak, and one revision averaged uninitialised memory. Delete them once you are satisfied
with this.

The first cell compares your local copy of the reader against the repository copy when it can reach
it, and says so when they differ. Drift becomes visible instead of silent.


In [ ]:
import glob
import hashlib
import importlib.util
import os
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- configuration
# Directories to read. Add your own. These are read-only; nothing is written back to them.
INPUT_DIRS = [
    "/mnt/c/Nocterra OneDrive/OneDrive - Nocterra/01 Projects/N043 28South"
    "/N04301 Griffin Sports Complex Lighting Assessment/03 Project Info/ies",
]
OUTPUT_DIR = "converted_lop"
PATTERNS = ("*.ies", "*.IES", "*.ldt", "*.LDT", "*.lop", "*.LOP")

# The repository copy of the reader, used only to warn about drift. Set to None to skip the check.
REPO_READER = os.path.expanduser("~/git/noc-illumina/illum/AngularPowerDistribution.py")

# ---------------------------------------------------------------- load the reader
def load_reader():
    """Return the photometry reader, preferring a copy beside this notebook."""
    local = os.path.join(os.path.abspath(""), "AngularPowerDistribution.py")
    if os.path.exists(local):
        spec = importlib.util.spec_from_file_location("AngularPowerDistribution", local)
        module = importlib.util.module_from_spec(spec)
        sys.modules["AngularPowerDistribution"] = module
        spec.loader.exec_module(module)
        return module, local
    import illum.AngularPowerDistribution as module
    return module, module.__file__


def fingerprint(path):
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()[:12]


APD, APD_PATH = load_reader()
print(f"reader   {APD_PATH}")
print(f"         {fingerprint(APD_PATH)}  modified {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(APD_PATH)))}")
if REPO_READER and os.path.exists(REPO_READER) and os.path.realpath(REPO_READER) != os.path.realpath(APD_PATH):
    if fingerprint(REPO_READER) == fingerprint(APD_PATH):
        print("         matches the repository copy")
    elif os.path.getmtime(REPO_READER) > os.path.getmtime(APD_PATH):
        print(f"         the repository copy {fingerprint(REPO_READER)} is NEWER")
        print(f"         copy it over this one:  {REPO_READER}")
    else:
        print(f"         the repository copy {fingerprint(REPO_READER)} is older, so this copy is ahead")
        print("         put your changes back into the repository")

# ---------------------------------------------------------------- helpers
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

ANG = np.arange(181.0)
RING = 2 * np.pi * np.diff(-np.cos(np.deg2rad(APD.mids(ANG))))


def find_files(dirs=None):
    """Every photometry file in the input directories, each listed once."""
    seen, out = set(), []
    for d in dirs or INPUT_DIRS:
        for pat in PATTERNS:
            for f in glob.glob(os.path.join(d, pat)):
                real = os.path.realpath(f)
                if real not in seen:
                    seen.add(real)
                    out.append(f)
    return sorted(out)


def profile_on_grid(apd):
    """The azimuth-averaged profile, resampled onto 0 to 180 degrees from nadir."""
    va = np.asarray(apd.vertical_angles, dtype=float)
    pr = np.asarray(apd.vertical_profile(), dtype=float)
    order = np.argsort(va)
    return np.interp(ANG, va[order], pr[order], left=0.0, right=0.0)


def uplight_ratio(prof):
    """The fraction of flux that leaves above the horizon."""
    flux = np.asarray(prof, dtype=float) * RING
    total = flux.sum()
    return flux[ANG > 90].sum() / total if total else np.nan


print(f"\n{len(find_files())} files found")


## 1. Inventory

One row per file.

`flux check` divides the integral of the distribution by the luminous flux the file declares. It should
sit at 1.00. A dash means the file uses absolute photometry and declares no flux, so there is nothing
to check against.

In [ ]:
rows = []
for f in find_files():
    rec = {"file": os.path.basename(f), "format": os.path.splitext(f)[1].lstrip(".").lower()}
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            apd = APD.read(f)
        prof = profile_on_grid(apd)
        flux = float(np.sum(apd.vertical_profile(integrated=True)))
        declared = apd.lumens if apd.lumens and apd.lumens > 0 else np.nan
        rec.update({
            "type": "CBA"[apd.type - 1] if apd.type in (1, 2, 3) else str(apd.type),
            "planes": len(apd.horizontal_angles),
            "angles": len(apd.vertical_angles),
            "declared lm": declared,
            "integral lm": flux,
            "flux check": flux / declared if declared == declared else np.nan,
            "uplight %": 100 * uplight_ratio(prof),
            "status": "ok",
            "notes": "; ".join(str(c.message)[:50] for c in caught),
        })
    except Exception as exc:
        rec["status"] = f"{type(exc).__name__}: {exc}"[:70]
    rows.append(rec)

inventory = pd.DataFrame(rows).reindex(columns=[
    "file", "format", "type", "planes", "angles",
    "declared lm", "integral lm", "flux check", "uplight %", "status", "notes",
])
try:
    display(inventory.style.format(
        {"declared lm": "{:,.0f}", "integral lm": "{:,.0f}", "flux check": "{:.3f}", "uplight %": "{:.3f}"},
        na_rep="-",
    ))
except (ImportError, AttributeError):   # styling needs jinja2; plain text works everywhere
    with pd.option_context("display.width", 220, "display.max_colwidth", 38):
        print(inventory.round({"declared lm": 0, "integral lm": 0, "flux check": 3, "uplight %": 3})
                       .to_string(index=False, na_rep="-"))


## 2. Choose one luminaire

Set `PICK` to a row number from the table above, or put part of a file name in `NAME`.

In [ ]:
PICK = 0
NAME = ""

files = find_files()
if NAME:
    files = [f for f in files if NAME.lower() in os.path.basename(f).lower()] or files
path = files[PICK]
apd = APD.read(path)
prof = profile_on_grid(apd)

print(os.path.basename(path))
print(f"  photometric type {'CBA'[apd.type - 1]}")
print(f"  {len(apd.vertical_angles)} vertical angles, {apd.vertical_angles[0]:g} to {apd.vertical_angles[-1]:g} deg from nadir")
print(f"  {len(apd.horizontal_angles)} C-planes, {apd.horizontal_angles[0]:g} to {apd.horizontal_angles[-1]:g} deg")
if apd.lumens and apd.lumens > 0:
    print(f"  declared flux {apd.lumens:,.0f} lm")
else:
    print("  absolute photometry, no declared flux")
print(f"  uplight {100 * uplight_ratio(prof):.3f} % of total flux")

## 3. Look at it

Four views of the same luminaire.

- **Polar.** The diagram a data sheet shows, with nadir pointing down.
- **Full distribution.** Azimuthal asymmetry shows as banding across the plot.
- **Plane spread.** Every C-plane over the azimuth average. This shows how wrong one plane can be.
- **Uplight detail.** A log scale above the horizon, with the floor that two decimal places imposes.

In [ ]:
fig = plt.figure(figsize=(11.5, 9))

ax = fig.add_subplot(221, polar=True)
colour = "C0"
for sign in (1, -1):
    ax.plot(np.deg2rad(sign * ANG), prof, lw=2, color=colour)
    ax.fill(np.deg2rad(sign * ANG), prof, alpha=0.2, color=colour)
ax.set_theta_zero_location("S")
ax.set_theta_direction(1)
ax.set_title("azimuth-averaged intensity (cd)", pad=16)

ax = fig.add_subplot(222)
web = apd.interpolate(step=2)
mesh = ax.pcolormesh(web.horizontal_angles, web.vertical_angles, web.data, shading="auto", cmap="inferno")
ax.axhline(90, color="w", ls="--", lw=1)
ax.set_xlabel("azimuth C (deg)")
ax.set_ylabel("angle from nadir (deg)")
ax.set_title("full distribution; dashed line is the horizon")
ax.grid(False)
fig.colorbar(mesh, ax=ax, label="cd")

ax = fig.add_subplot(223)
for i in range(len(apd.horizontal_angles)):
    ax.plot(apd.vertical_angles, apd.data[:, i], color="0.75", lw=0.7,
            label="individual C-planes" if i == 0 else None)
ax.plot(ANG, prof, "k", lw=2, label="azimuth average, what the model uses")
ax.plot(apd.vertical_angles, apd.data[:, 0], "r", lw=1.3, label="first C-plane, what the old notebook used")
ax.axvline(90, color="C0", ls="--", lw=1)
ax.set_xlabel("angle from nadir (deg)")
ax.set_ylabel("intensity (cd)")
ax.set_title("does the plane choice matter?")
ax.legend(fontsize=8)

ax = fig.add_subplot(224)
peak = prof.max() or 1.0
ax.semilogy(ANG, np.maximum(prof / peak, 1e-9), lw=2, label="full precision")
ax.semilogy(ANG, np.maximum(np.round(prof / peak, 2), 1e-9), lw=1.2, ls="--", label="rounded to 2 dp, the old format")
ax.axvline(90, color="C0", ls="--", lw=1)
ax.axhline(0.005, color="r", lw=0.8)
ax.set_ylim(1e-6, 2)
ax.set_xlabel("angle from nadir (deg)")
ax.set_ylabel("intensity / peak")
ax.set_title("uplight sits above 90 deg; red line is the 2 dp floor")
ax.legend(fontsize=8)

fig.suptitle(os.path.basename(path), y=0.997)
fig.tight_layout()

## 4. Checks

`PASS` means the file behaves as its own header says it should. `CHECK` marks something to look at
by eye before you trust the conversion.

In [ ]:
checks = []

flux = float(np.sum(apd.vertical_profile(integrated=True)))
if apd.lumens and apd.lumens > 0:
    ratio = flux / apd.lumens
    checks.append(("integral reproduces the declared flux", f"{ratio:.4f}", abs(ratio - 1) < 0.01))
else:
    checks.append(("declared flux", "absolute photometry, nothing to compare", True))

va = np.asarray(apd.vertical_angles, dtype=float)
checks.append(("vertical angles ascend", f"{va[0]:g} to {va[-1]:g}", bool(np.all(np.diff(va) > 0))))
checks.append(("intensities are finite and positive", f"min {np.nanmin(apd.data):.4g}",
               bool(np.all(np.isfinite(apd.data)) and np.nanmin(apd.data) >= 0)))

up = uplight_ratio(prof)
peak = prof.max() or 1.0
up_rounded = uplight_ratio(np.round(prof / peak, 2))
checks.append(("uplight survives 2 dp rounding", f"{100 * up:.3f} % becomes {100 * up_rounded:.3f} %",
               up == 0 or up_rounded > 0))

order = np.argsort(va)
plane0 = np.interp(ANG, va[order], np.asarray(apd.data)[order, 0], left=0.0, right=0.0)
up0 = uplight_ratio(plane0)
checks.append(("one C-plane represents the whole luminaire", f"uplight {100 * up0:.3f} % against {100 * up:.3f} %",
               bool(abs(up0 - up) <= 0.1 * max(up, 1e-12))))

for name, value, ok in checks:
    print(f"  [{'PASS' if ok else 'CHECK'}]  {name:44s} {value}")

## 5. Cross-format agreement

When the same luminaire ships as both IES and EULUMDAT, the two conversions must land on the same
profile. Any real difference is a bug in one of the readers.

In [ ]:
pair = [f for f in find_files() if "vfl530" in os.path.basename(f).lower()]

profs = {}
for f in pair:
    ext = os.path.splitext(f)[1].lstrip(".").lower()
    try:
        p = profile_on_grid(APD.read(f))
        profs[ext] = p / (p.max() or 1.0)
    except Exception as exc:
        print(f"{os.path.basename(f)}: {type(exc).__name__}: {exc}")

if len(profs) >= 2:
    fig, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4))
    for ext, p in profs.items():
        left.plot(ANG, p, lw=2, label=f"{ext}, uplight {100 * uplight_ratio(p):.3f} %")
    left.axvline(90, color="0.5", ls="--")
    left.set_xlabel("angle from nadir (deg)")
    left.set_ylabel("intensity / peak")
    left.set_title("same luminaire, two formats")
    left.legend()

    keys = list(profs)
    diff = profs[keys[0]] - profs[keys[1]]
    right.plot(ANG, diff, lw=1.5)
    right.axvline(90, color="0.5", ls="--")
    right.set_xlabel("angle from nadir (deg)")
    right.set_ylabel(f"{keys[0]} minus {keys[1]}")
    right.set_title(f"largest difference {np.abs(diff).max():.2e} of peak")
    fig.tight_layout()
else:
    print("Need two readable files for one luminaire. Point `pair` at them.")


## 6. Export

Writes one `.lop` per input at full precision into `OUTPUT_DIR`. It never writes back into the folder
it read from, so a client directory stays untouched.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

written, failed = [], []
for f in find_files():
    ext = os.path.splitext(f)[1].lower()
    if ext == ".lop":
        continue
    stem = os.path.splitext(os.path.basename(f))[0]
    out = os.path.join(OUTPUT_DIR, stem + ".lop")
    if out in written:          # the same luminaire in two formats must not overwrite itself
        out = os.path.join(OUTPUT_DIR, f"{stem}_{ext.lstrip('.')}.lop")
    try:
        APD.to_txt(out, APD.read(f))
        written.append(out)
    except Exception as exc:
        failed.append((os.path.basename(f), f"{type(exc).__name__}: {exc}"[:70]))

print(f"wrote {len(written)} files into {os.path.abspath(OUTPUT_DIR)}")
for name in written:
    print("   ", os.path.basename(name))
if failed:
    print(f"\n{len(failed)} failed:")
    for name, err in failed:
        print("   ", name, "|", err)
